# Silver — Duplicate / Near-duplicate detection (TF‑IDF vectors + cosine similarity)

Mục tiêu: phát hiện các bài có nội dung trùng / gần trùng trong JSON ở lớp **silver**.

Công thức cosine similarity:
$$\cos(\theta)=\frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|}$$

Quy tắc: nếu similarity > `SIM_THRESHOLD` (mặc định 0.9) ⇒ loại bài bị trùng.

**Input/Output (silver 1/2/3):**
- Input: `data/<YEAR>/<BANK>/silver/1/article_texts_*.json` (hoặc fallback legacy `.../silver/article_texts_*.json`)
- Output: `data/<YEAR>/<BANK>/silver/2/<input>_dedup.json` + file log `*_duplicates_log.csv`

In [1]:
# 1) Cài đặt thư viện (chạy 1 lần nếu thiếu)
%pip -q install numpy scikit-learn pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [4]:
# 3) Cấu hình: đường dẫn + ngưỡng
# --- CẤU HÌNH NĂM VÀ NGÂN HÀNG CẦN XỬ LÝ ---
TARGET_YEAR = "2023"
TARGET_BANK = "Vietcombank" 
# ------------------------------------------

In [5]:
# 2) Import
from __future__ import annotations

import json
import re
import unicodedata
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [6]:

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() in {"thread_1", "thread_2"}:
    # Tự động root về NCKH/Thread_1 nếu bị lùi
    # Do cấu trúc data đang nằm đúng trong Thread_1 nên ta không cần back ra NCKH
    pass

DATA_ROOT = PROJECT_ROOT / "data" / str(TARGET_YEAR)

# Silver folder structure:
# - silver/1: JSON sau khi chạy crawl_contents_from_filter_excels.py
# - silver/2: JSON sau khi loại duplicate/near-duplicate (notebook này)
SILVER_IN = "silver/1"
SILVER_OUT = "silver/2"

# Nếu muốn xử lý 1 file cụ thể, set path ở đây (vd: Path(r"data/2023/BIDV/silver/1/article_texts_....json"))
INPUT_JSON_PATH: Path | None = None

# Near-duplicate threshold
SIM_THRESHOLD = 0.90

# Cosine distance = 1 - similarity → radius cho radius_neighbors
RADIUS = 1.0 - SIM_THRESHOLD

# TF-IDF config (char n-grams thường ổn với tiếng Việt khi không muốn tokenizer riêng)
TFIDF_ANALYZER = "char_wb"
TFIDF_NGRAM_RANGE = (3, 5)
TFIDF_MIN_DF = 1

print("DATA_ROOT:", DATA_ROOT)
print("TARGET_BANK:", TARGET_BANK)
print("SILVER_IN:", SILVER_IN)
print("SILVER_OUT:", SILVER_OUT)
print("SIM_THRESHOLD:", SIM_THRESHOLD, "=> radius:", RADIUS)

DATA_ROOT: d:\NCKH\Thread_1\data\2023
TARGET_BANK: Vietcombank
SILVER_IN: silver/1
SILVER_OUT: silver/2
SIM_THRESHOLD: 0.9 => radius: 0.09999999999999998


In [7]:
# 4) Load JSON + chuẩn hoá schema
@dataclass
class Record:
    bank: str
    idx: int
    link: str
    title: str
    content: str
    source: str


def infer_bank_from_json_path(p: Path) -> str:
    parts = list(p.parts)
    lower = [x.lower() for x in parts]
    if "silver" in lower:
        i = lower.index("silver")
        if i >= 1:
            return parts[i - 1]
    # Fallback
    try:
        return p.parent.parent.name
    except Exception:
        return ""


def find_candidate_json_files(data_root: Path, stage_in: str, target_bank: str = "*") -> list[Path]:
    """Find candidate silver JSON files.

    New layout: data/<YEAR>/<BANK>/silver/1/article_texts_*.json
    or data/<YEAR>/silver/1/article_texts_*.json
    """
    files: list[Path] = []
    bank_path = "**" if target_bank == "*" else target_bank
    
    # Quét theo bank
    files.extend([p for p in data_root.glob(f"{bank_path}/{stage_in}/article_texts_*.json") if p.is_file()])
    
    # Fallback: if stage_in is silver/1 and nothing found, also look for old silver/*.json
    if not files and stage_in.replace("\\", "/") in {"silver/1", "silver"}:
        files.extend([p for p in data_root.glob(f"{bank_path}/silver/article_texts_*.json") if p.is_file()])
    return sorted(files)


def pick_latest_per_bank(files: list[Path]) -> list[Path]:
    # nếu 1 bank có nhiều file theo timestamp, lấy file mới nhất theo mtime
    by_bank: dict[str, Path] = {}
    for p in files:
        bank = infer_bank_from_json_path(p)
        cur = by_bank.get(bank)
        if cur is None or p.stat().st_mtime > cur.stat().st_mtime:
            by_bank[bank] = p
    return sorted(by_bank.values(), key=lambda x: infer_bank_from_json_path(x))


def load_bank_records_from_json(path: Path) -> list[Record]:
    obj = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(obj, dict):
        raise ValueError("JSON root must be dict {bank: [items]}")
    records: list[Record] = []
    for bank, items in obj.items():
        if not isinstance(items, list):
            continue
        for i, it in enumerate(items):
            if not isinstance(it, dict):
                continue
            link = str(it.get("link") or "")
            title = str(it.get("title") or "")
            content = str(it.get("content") or "")
            source = str(it.get("source") or "")
            records.append(Record(bank=str(bank), idx=i, link=link, title=title, content=content, source=source))
    return records


def load_original_json(path: Path) -> dict[str, list[dict[str, Any]]]:
    obj = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(obj, dict):
        raise ValueError("JSON root must be dict")
    return obj

In [8]:
# 5) Tiền xử lý text
_WS_RE = re.compile(r"\s+")


def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.lower().strip()
    text = _WS_RE.sub(" ", text)
    # bỏ ký tự điều khiển cơ bản
    text = "".join(ch for ch in text if ch.isprintable() or ch in "\n\t")
    return text.strip()


def build_doc(record: Record) -> str:
    title = normalize_text(record.title)
    content = normalize_text(record.content)
    doc = (title + "\n" + content).strip() if content else title
    return doc

In [9]:
# 6) Vector hoá TF‑IDF (sparse)
def build_tfidf_matrix(docs: list[str]) -> tuple[Any, TfidfVectorizer]:
    vec = TfidfVectorizer(
        analyzer=TFIDF_ANALYZER,
        ngram_range=TFIDF_NGRAM_RANGE,
        min_df=TFIDF_MIN_DF,
        dtype=np.float32,
    )
    X = vec.fit_transform(docs)
    return X, vec

In [10]:
# 7) Tìm near-duplicates bằng cosine similarity > threshold
class DSU:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.size = [1] * n
    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a: int, b: int) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]


def find_duplicate_pairs(X, sim_threshold: float) -> list[tuple[int, int, float]]:
    # cosine distance = 1 - similarity
    radius = 1.0 - sim_threshold
    nn = NearestNeighbors(metric='cosine', algorithm='brute')
    nn.fit(X)
    dists, inds = nn.radius_neighbors(X, radius=radius, return_distance=True)
    pairs: list[tuple[int, int, float]] = []
    for i, (nbrs, ds) in enumerate(zip(inds, dists)):
        for j, d in zip(nbrs, ds):
            if j <= i:
                continue
            sim = 1.0 - float(d)
            if sim >= sim_threshold:
                pairs.append((i, int(j), sim))
    return pairs

In [11]:
# 8) Quy tắc giữ/xoá + ghi file output (silver/2)
def choose_keep_and_drop(
    records: list[Record],
    pairs: list[tuple[int, int, float]],
) -> tuple[set[int], dict[int, int]]:
    """Trong mỗi cụm duplicate, giữ bản tốt hơn (ưu tiên content dài), drop các bản còn lại.

    Returns:
      - drop_set: tập index bị xoá
      - keep_of: mapping index -> index được giữ trong cùng cụm
    """
    n = len(records)
    dsu = DSU(n)
    for i, j, _sim in pairs:
        dsu.union(i, j)
    groups: dict[int, list[int]] = {}
    for i in range(n):
        root = dsu.find(i)
        groups.setdefault(root, []).append(i)

    def _score(i: int) -> tuple[int, int, int, int]:
        # Prefer records that actually have content, then longer content, then longer title, then smaller index.
        content = normalize_text(records[i].content)
        title = normalize_text(records[i].title)
        has_content = 1 if content else 0
        return (has_content, len(content), len(title), -i)

    drop: set[int] = set()
    keep_of: dict[int, int] = {}
    for members in groups.values():
        if len(members) <= 1:
            keep_of[members[0]] = members[0]
            continue
        keep = max(members, key=_score)
        for m in members:
            keep_of[m] = keep
            if m != keep:
                drop.add(m)
    return drop, keep_of


def infer_bank_root_from_input_path(input_path: Path) -> Path:
    """Given .../data/<YEAR>/<BANK>/silver/1/<file>.json -> return .../data/<YEAR>/<BANK>."""
    parts = list(input_path.parts)
    lower = [x.lower() for x in parts]
    if "silver" in lower:
        i = lower.index("silver")
        return Path(*parts[:i])
    # Fallback
    return input_path.parent.parent


def ensure_unique_path(path: Path) -> Path:
    if not path.exists():
        return path
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    return path.with_name(f"{path.stem}_{ts}{path.suffix}")


def write_json_atomic(path: Path, obj: Any) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path = ensure_unique_path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)
    return path


def make_output_paths(input_path: Path, stage_out: str) -> tuple[Path, Path]:
    bank_root = infer_bank_root_from_input_path(input_path)
    out_dir = bank_root / stage_out
    out_dir.mkdir(parents=True, exist_ok=True)
    out_json = out_dir / f"{input_path.stem}_dedup{input_path.suffix}"
    out_log = out_dir / f"{input_path.stem}_dedup_duplicates_log.csv"
    return out_json, out_log

In [12]:
# 9) Chạy dedupe: theo từng bank trong 1 file JSON (input: silver/1, output: silver/2)
def dedupe_one_json_file(input_path: Path, stage_out: str) -> dict[str, Any]:
    original = load_original_json(input_path)
    out: dict[str, list[dict[str, Any]]] = {}
    log_rows: list[dict[str, Any]] = []
    total_before = 0
    total_after = 0
    clusters_count = 0
    max_cluster_size = 1
    
    for bank, items in original.items():
        if not isinstance(items, list) or not items:
            out[bank] = items if isinstance(items, list) else []
            continue
        # chuẩn hoá list dict
        norm_items = [it for it in items if isinstance(it, dict)]
        total_before += len(norm_items)
        records = [
            Record(
                bank=str(bank),
                idx=i,
                link=str(it.get("link") or ""),
                title=str(it.get("title") or ""),
                content=str(it.get("content") or ""),
                source=str(it.get("source") or ""),
            )
            for i, it in enumerate(norm_items)
        ]
        docs = [build_doc(r) for r in records]
        nonempty_docs = [d for d in docs if d.strip()]
        if len(nonempty_docs) <= 1:
            # Không đủ text để so sánh / hoặc toàn trống → giữ nguyên
            out[bank] = norm_items
            total_after += len(norm_items)
            print(f"- {bank}: {len(norm_items)} -> {len(norm_items)} (skip: not enough text)")
            continue
        # Guard: nếu mọi doc quá ngắn so với ngram_range => TF-IDF empty vocabulary
        if max(len(d) for d in nonempty_docs) < TFIDF_NGRAM_RANGE[0]:
            out[bank] = norm_items
            total_after += len(norm_items)
            print(f"- {bank}: {len(norm_items)} -> {len(norm_items)} (skip: docs too short for ngrams)")
            continue
        
        try:
            X, _vec = build_tfidf_matrix(docs)
        except ValueError as e:
            # e.g., empty vocabulary
            out[bank] = norm_items
            total_after += len(norm_items)
            print(f"- {bank}: {len(norm_items)} -> {len(norm_items)} (skip: {e})")
            continue
        pairs = find_duplicate_pairs(X, SIM_THRESHOLD)
        drop_set, keep_of = choose_keep_and_drop(records, pairs)
        kept_items = [norm_items[i] for i in range(len(norm_items)) if i not in drop_set]
        out[bank] = kept_items
        total_after += len(kept_items)
        removed_n = len(norm_items) - len(kept_items)
        print(f"- {bank}: {len(norm_items)} -> {len(kept_items)} (removed {removed_n})")
        
        # Log only actual removed items, with similarity vs kept representative (computed from TF-IDF)
        for j in sorted(drop_set):
            i = keep_of.get(j, j)
            try:
                sim = float((X[j] @ X[i].T).A[0][0])
            except Exception:
                sim = None
            log_rows.append({
                "bank": bank,
                "keep_i": i,
                "drop_j": j,
                "similarity": sim,
                "keep_link": records[i].link,
                "drop_link": records[j].link,
                "keep_title": records[i].title[:200],
                "drop_title": records[j].title[:200],
            })
        
        # stats cụm (ước lượng)
        if drop_set:
            dsu = DSU(len(records))
            for i, j, _sim in pairs:
                dsu.union(i, j)
            groups: dict[int, list[int]] = {}
            for i in range(len(records)):
                groups.setdefault(dsu.find(i), []).append(i)
            sizes = [len(m) for m in groups.values() if len(m) > 1]
            clusters_count += len(sizes)
            if sizes:
                max_cluster_size = max(max_cluster_size, max(sizes))
    
    out_json_path, out_log_path = make_output_paths(input_path, stage_out=stage_out)
    out_json_path = write_json_atomic(out_json_path, out)
    log_df = pd.DataFrame(log_rows)
    log_csv_path = None
    if not log_df.empty:
        log_df = log_df.sort_values(["bank", "similarity"], ascending=[True, False], na_position="last")
        log_df.to_csv(out_log_path, index=False, encoding="utf-8-sig")
        log_csv_path = str(out_log_path)
    return {
        "file_in": str(input_path),
        "file_out": str(out_json_path),
        "log_csv": log_csv_path,
        "before": total_before,
        "after": total_after,
        "removed": total_before - total_after,
        "clusters": clusters_count,
        "max_cluster_size": max_cluster_size,
    }

In [13]:
# 10) Chọn input file và chạy
if INPUT_JSON_PATH is None:
    candidates = find_candidate_json_files(DATA_ROOT, stage_in=SILVER_IN, target_bank=TARGET_BANK)
    latest_files = pick_latest_per_bank(candidates)
    print("Auto-pick latest silver JSON per bank:", len(latest_files))
    for p in latest_files:
        print("-", p)
    target_files = latest_files
else:
    target_files = [INPUT_JSON_PATH]

results = []
for p in target_files:
    print("\n=== DEDUPE FILE ===")
    print("Input:", p)
    res = dedupe_one_json_file(p, stage_out=SILVER_OUT)
    results.append(res)
    print("Result:", res)

pd.DataFrame(results)

Auto-pick latest silver JSON per bank: 1
- d:\NCKH\Thread_1\data\2023\Vietcombank\silver\1\article_texts_20260404_203531.json

=== DEDUPE FILE ===
Input: d:\NCKH\Thread_1\data\2023\Vietcombank\silver\1\article_texts_20260404_203531.json
- Vietcombank: 113 -> 106 (removed 7)
Result: {'file_in': 'd:\\NCKH\\Thread_1\\data\\2023\\Vietcombank\\silver\\1\\article_texts_20260404_203531.json', 'file_out': 'd:\\NCKH\\Thread_1\\data\\2023\\Vietcombank\\silver\\2\\article_texts_20260404_203531_dedup.json', 'log_csv': 'd:\\NCKH\\Thread_1\\data\\2023\\Vietcombank\\silver\\2\\article_texts_20260404_203531_dedup_duplicates_log.csv', 'before': 113, 'after': 106, 'removed': 7, 'clusters': 5, 'max_cluster_size': 4}


,file_in,file_out,log_csv,before,after,removed,clusters,max_cluster_size
0,d:\NCKH\Thread_1\data\2023\Vietcombank\silver\...,d:\NCKH\Thread_1\data\2023\Vietcombank\silver\...,d:\NCKH\Thread_1\data\2023\Vietcombank\silver\...,113,106,7,5,4


In [14]:
# 11) Spot-check nhanh: xem vài dòng log duplicate (nếu có)
for r in results:
    log_csv = r.get('log_csv')
    if not log_csv:
        continue
    log_path = Path(log_csv)
    if log_path.exists():
        df_log = pd.read_csv(log_path)
        display(df_log.head(20))

,bank,keep_i,drop_j,similarity,keep_link,drop_link,keep_title,drop_title
0,Vietcombank,59,8,NaN,https://cafef.vn/vcb-digibot-tro-ly-ao-thong-m...,https://vnexpress.net/vietcombank-toi-uu-hieu-...,VCB Digibot – Trợ lý ảo thông minh của Vietcom...,Vietcombank tối ưu hiệu suất nhờ trợ lý ảo AI ...
1,Vietcombank,0,12,NaN,https://tapchinganhang.gov.vn/ra-mat-bo-ba-san...,https://vnexpress.net/ra-mat-bo-ba-san-pham-th...,Ra mắt bộ ba sản phẩm thẻ Vietcombank thương h...,Ra mắt bộ ba sản phẩm thẻ Vietcombank Visa - V...
2,Vietcombank,40,17,NaN,https://baochinhphu.vn/vietcombank-ra-mat-the-...,https://vnexpress.net/vietcombank-ra-mat-the-g...,Vietcombank ra mắt thẻ 'Ghi nợ quốc tế VCB Dig...,Vietcombank ra mắt thẻ ghi nợ quốc tế VCB Digi...
3,Vietcombank,38,22,NaN,https://baochinhphu.vn/chia-khoa-de-vietcomban...,https://vnexpress.net/dau-an-cua-vietcombank-t...,'Chìa khóa' để Vietcombank chinh phục đỉnh cao...,Dấu ấn của Vietcombank trên hành trình chuyển ...
4,Vietcombank,0,26,NaN,https://tapchinganhang.gov.vn/ra-mat-bo-ba-san...,https://vnexpress.net/vietcombank-ra-mat-bo-ba...,Ra mắt bộ ba sản phẩm thẻ Vietcombank thương h...,Vietcombank ra mắt bộ ba thẻ Visa mới - VnExpress
5,Vietcombank,0,32,NaN,https://tapchinganhang.gov.vn/ra-mat-bo-ba-san...,https://baochinhphu.vn/ra-mat-bo-ba-san-pham-t...,Ra mắt bộ ba sản phẩm thẻ Vietcombank thương h...,Ra mắt Bộ ba sản phẩm thẻ Vietcombank thương h...
6,Vietcombank,107,33,NaN,https://cafef.vn/vietcombank-ra-mat-website-ho...,https://baochinhphu.vn/vietcombank-chinh-thuc-...,Vietcombank ra mắt website hoàn toàn mới: Hiện...,Vietcombank chính thức ra mắt website mới - ba...
